# Bank customer churn — K-Nearest Neighbors (KNN)

Supervised binary classification on preprocessed splits (`x_train.csv`, `x_test.csv`, `y_train.csv`, `y_test.csv`). Target: customer churn (`Exited`).

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
)


def resolve_paths():
    colab = Path("/content/drive/MyDrive/bank-churn-dataset/data")
    local = Path("..") / "data"
    if colab.joinpath("x_train.csv").exists():
        data_dir, report_dir = colab, colab.parent / "report"
    else:
        data_dir, report_dir = local.resolve(), (Path("..") / "report").resolve()
    report_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, report_dir


DATA_DIR, REPORT_DIR = resolve_paths()

## Data

In [ ]:
X_train = pd.read_csv(DATA_DIR / "x_train.csv")
X_test = pd.read_csv(DATA_DIR / "x_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape, X_test.shape)
print(f"Churn rate train/test: {y_train.mean():.3f}, {y_test.mean():.3f}")

## Model

`KNeighborsClassifier` with `GridSearchCV` (stratified 5-fold, `scoring='roc_auc'`): `n_neighbors`, `weights`, `p`.

In [ ]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15, 21, 25],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid.fit(X_train, y_train)

print("Best CV ROC-AUC:", grid.best_score_)
print("Best params:", grid.best_params_)
knn_model = grid.best_estimator_

## Test metrics

In [ ]:
y_pred = knn_model.predict(X_test)
y_pred_proba = knn_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("KNN — test set")
print(f"Accuracy  {accuracy:.4f}")
print(f"Precision {precision:.4f}")
print(f"Recall    {recall:.4f}")
print(f"F1        {f1:.4f}")
print(f"ROC-AUC   {roc_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=["Not churned", "Churned"]))

## Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Not churned", "Churned"]).plot(
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title("KNN — confusion matrix")
plt.tight_layout()
plt.savefig(REPORT_DIR / "knn_confusion_matrix.png", dpi=150)
plt.show()

## ROC curve

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, name="KNN", ax=ax, color="#2ca02c")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_title("KNN — ROC curve")
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "knn_roc_curve.png", dpi=150)
plt.show()

## CV ROC-AUC vs. k

In [ ]:
cv_df = pd.DataFrame(
    {
        "n_neighbors": [p["n_neighbors"] for p in grid.cv_results_["params"]],
        "weights": [p["weights"] for p in grid.cv_results_["params"]],
        "p": [p["p"] for p in grid.cv_results_["params"]],
        "mean_test_roc_auc": grid.cv_results_["mean_test_score"],
    }
)
best_w, best_p = grid.best_params_["weights"], grid.best_params_["p"]
sub = cv_df[(cv_df["weights"] == best_w) & (cv_df["p"] == best_p)].sort_values("n_neighbors")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sub["n_neighbors"], sub["mean_test_roc_auc"], marker="o")
ax.set_xlabel("k (n_neighbors)")
ax.set_ylabel("Mean CV ROC-AUC")
ax.set_title(f"CV ROC-AUC vs k (weights={best_w!r}, p={best_p})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / "knn_cv_roc_auc_vs_k.png", dpi=150)
plt.show()